# 👩‍🎓 Child Speech Recognition for Early-Grade Reading Assessments
_A PhD Theory + Lab Notebook with Presentation, Diagrams, Results & Code_

**Robert-Florin Simion | UPB ETTI**
Supervisor: Prof. Dr. Ing. Dragoș Burileanu

> 📋 Welcome! This notebook merges all theory, presentation, experiments, code labs, and charts with design, emojis, tables, and explanations. Use it for your documentation.


## 📊 Motivation & Problem
- **Existing ASR models:** Optimized for adults
- **Children's speech characteristics:** Higher pitch, less fluency
- **Goal:** Automated reading assessment for education
- **Gap:** WER on child speech ~12–15% vs <3% for adults
---

## 🎯 2. Thesis Objectives and Approach

- **General Objective**
    - Develop robust child ASR systems for educational purposes
- **Specific Objectives**
    - Match adult-level accuracy
    - Adapt ASR for phonics, decoding, syllable tasks
    - Automate scoring for reading assessments
- **Expected Deliverables**:
    - Web/mobile demo app
    - Publications: Interspeech, ICASSP...

## 🧪 3. Research Workflow & Methodology

```mermaid graph TD;
    A[Literature Review] --> B[Model Development: wav2vec2 & Whisper];
    B --> C[Data Preparation: MyST, PFSTAR, etc];
    C --> D[Experimentation: Pretrain & Finetune];
    D --> E[Evaluation: WER & Cross-validation];
    E --> F[Deployment: Apps, Papers];
```

Timeline:
- Year 1: Literature, datasets, baseline
- Year 2: Advanced models
- Year 3: Romanian corpus 
- Year 4: Deploy, write-up, publish
---

## 📚 4. Literature Review – Key Papers
### Jain et al. (2023a, IEEE Access)
• **wav2vec2**: Self-supervised, 7.42% WER (MyST)
### Jain et al. (2023b, Interspeech)
• **Whisper**: Weakly-supervised, 12.28% WER (MyST), better zero-shot
### Jain et al. (2024, IEEE Access)
• **Whisper**: Non-native scenarios, 29-89% WER improvement

---

## 🏗️ 5. Architectures

*Generics*
**🗣️ What is ASR? (Automatic Speech Recognition)**
Automatic Speech Recognition (ASR) systems convert spoken language (audio) into written text. They power virtual assistants, voice search, and transcription.

***Pipeline***
- Audio Input: Microphone or audio file
- Feature Extraction: Converts audio waveform into numerical features (MFCCs, Mel spectrogram, or neural features)
- Acoustic Model: Neural network predicts speech sounds (phonemes, graphemes)
- Language Model: Optional, predicts word sequences for context
- Decoding: Finds the most probable transcription (CTC, beam search)
- Text Output: Final transcript
![asr_flowchart.png](charts/asr_flowchart.png)

**A. wav2vec2**
wav2vec2 is a neural network architecture for automatic speech recognition, developed by Facebook AI. It uses self-supervised learning—learning from stuff without labels!
Key Components:
- Feature Extractor: 7-layer CNN learns basic audio features from the raw waveform.
- Context Network (Transformer): Learns long-range relationships in audio.
- Quantization: Discretizes the audio features (using "codebooks").
- Pretraining: Learns to predict masked audio parts using contrastive loss, so model discovers good representations by itself.
- Finetuning: Adapts with labeled speech for transcription using CTC (Connectionist Temporal Classification).
---
**Wav2Vec2(Essentials):**
- CNN feature extractor, 7 blocks
- Transformer BASE: 12 layers, 768-dim
- Gumbel softmax quantization
![wav2vec2_architecture.png](charts/wav2vec2_architecture.png)
---
**B. Whisper**
- Encoder-decoder transformer, 2 CNN layers
- 80-channel log-Mel spectrogram
- Tiny-Medium-Large models
![whisper_architecture.png](charts/whisper_architecture.png)
---

## 📊 6. Performance Benchmarks

| Dataset | wav2vec2 | Whisper |
|---------|----------|---------|
| MyST | 7.42% | 12.28% |
| PFSTAR | 2.91% | 2.98% |
| CMU-Kids | 12.77% | 15.08% |

![chart.png](charts/chart.png)
---

In [2]:
## 🛠️ 7. Laboratory Examples: Code & Experiments
import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'

import sys
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import librosa
import torch
import torch.nn.functional as F

print(f"Python: {sys.executable}")
print(f"Torch version: {torch.__version__}")
# !{sys.executable} -m pip uninstall -y torch torchvision torchaudio
# !{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !{sys.executable} -m pip install transformers librosa torch soundfile openai-whisper jiwer
# print(f"Python executable: {sys.executable}")
# print(f"Python version: {sys.version}")

#### wav2vec2 Inference
# 'facebook/wav2vec2-large-960h-lv60-self' - pretrained sp-to-text
# CTC - Connectionist Temporal Classification (CTC) loss
model = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
# load processor (tokinezer & feature extractor) compatible with 'facebook/wav2vec2-large-960h-lv60-self'
# for audio processing & text decoding
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
# audio - waveform array, _ - SR of child.wav matching SR of 'facebook/wav2vec2-large-960h-lv60-self'
audio, _ = librosa.load('child.wav', sr=16000)
# converts the waveform for input of Wav2Vec2 model
# return_tensors - returns pytorch tensor
inputs = processor(audio, sampling_rate=16000, return_tensors='pt', padding=True)
# disable gradients (inference mode) 
# the logits - processed audio passed to get raw output logits (unnormalized predictions)
with torch.no_grad(): 
    logits = model(inputs.input_values).logits
# find most likely predicted tokens in sequence and for each time frame, selects highest score token
pred_ids = torch.argmax(logits, dim=-1)
# decodes predicted token IDs into readable text and returns first batch processed(the only, in this example)
transcription = processor.batch_decode(pred_ids)[0]
print(transcription)


# Confidence Scores for predictions
# softmax: Transforms raw logits into probabilities (across the vocabulary) for each frame
probs = F.softmax(logits, dim=-1)
# Get the maximum probability at each time frame, then average for a per-sample “overall confidence” score
confidence = probs.max(dim=-1).values.mean().item()

pred_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(pred_ids)[0]

print(f"Transcription: {transcription}")
print(f"Model confidence: {confidence:.2f}")

Python: E:\PHD\IA3\prezentare\demo-sr-ia3\.venv\Scripts\python.exe
Torch version: 2.9.0+cpu


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SHE JUMPED ROPE
Transcription: SHE JUMPED ROPE
Model confidence: 0.99


In [3]:
# Same but for mix dialects adult voice
model = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
# load processor (tokinezer & feature extractor) compatible with 'facebook/wav2vec2-large-960h-lv60-self'
# for audio processing & text decoding
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
# audio - waveform array, _ - SR of child.wav matching SR of 'facebook/wav2vec2-large-960h-lv60-self'
audio, _ = librosa.load('robert.wav', sr=16000)
# converts the waveform for input of Wav2Vec2 model
# return_tensors - returns pytorch tensor
inputs = processor(audio, sampling_rate=16000, return_tensors='pt', padding=True)
# disable gradients (inference mode) 
# the logits - processed audio passed to get raw output logits (unnormalized predictions)
with torch.no_grad(): 
    logits = model(inputs.input_values).logits




# Confidence Scores for predictions
# softmax: Transforms raw logits into probabilities (across the vocabulary) for each frame
probs = F.softmax(logits, dim=-1)
# Get the maximum probability at each time frame, then average for a per-sample “overall confidence” score
confidence = probs.max(dim=-1).values.mean().item()
# find most likely predicted tokens in sequence and for each time frame, selects highest score token
pred_ids = torch.argmax(logits, dim=-1)
# decodes predicted token IDs into readable text and returns first batch processed(the only, in this example)
transcription = processor.batch_decode(pred_ids)[0]

print(f"Transcription: {transcription}")
print(f"Model confidence: {confidence:.2f}")


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Transcription: HELLO EVERYONE I'M ROBERT AND I THINK THESE MODELS SHOULD HAVE GREAT ACCURACY FOR ADULT VOICE THAN MUSE ROMONA
Model confidence: 0.97


In [4]:
## Why child speech recognition is hard?

# 36months old inference 
model = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
audio, _ = librosa.load('36.wav', sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors='pt', padding=True)
with torch.no_grad(): 
    logits = model(inputs.input_values).logits
pred_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(pred_ids)[0]
print(transcription)


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


MAGICL CHRISMAS AN OLD ON HOUSE AD NOT GREULLY FILING NOT EVEN A MOUSE AN THE STOCKINGS WERE HUNG BY THE SUNLIT AN UMPE WAS KIS AN COKINAS ESUMABLY ERE


In [5]:
# Batch Transcription - With newborns & non-english native

model = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')

audio_files = ['child.wav', '24.wav', '36.wav', 'girl-12y.wav', 'boy-12y.wav']  # List of audio filenames
transcriptions = []

for fname in audio_files:
    audio, _ = librosa.load(fname, sr=16000)  
    inputs = processor(audio, sampling_rate=16000, return_tensors='pt', padding=True)
    with torch.no_grad():
        logits = model(inputs.input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)
    # torch.argmax: Selects the most probable token for each time frame based on logits
    transcription = processor.batch_decode(pred_ids)[0]
    transcriptions.append((fname, transcription))

for fname, transcription in transcriptions:
    print(f"{fname}: {transcription}")

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


child.wav: SHE JUMPED ROPE
24.wav: O NOSE  MYOI AVE NOT AN
36.wav: MAGICL CHRISMAS AN OLD ON HOUSE AD NOT GREULLY FILING NOT EVEN A MOUSE AN THE STOCKINGS WERE HUNG BY THE SUNLIT AN UMPE WAS KIS AN COKINAS ESUMABLY ERE
girl-12y.wav: THE NATURE IS VERY ATTRACTIVE AND FULL OF MY FAVORITE GREEN COLOR
boy-12y.wav: NATURE GIVES O FRUITS A VEGETABLES AND THE GRAINS WE TOR TO BE


In [6]:
# Handle noisy samples

import numpy as np

model = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')

audio, _ = librosa.load('robert.wav', sr=16000)
# Create Gaussian white noise with mean 0, standard deviation 0.02, same length as the audio
noise = np.random.normal(0, 0.02, audio.shape)
# Add the noise directly to the original audio to simulate a noisy environment
noisy_audio = audio + noise

inputs = processor(noisy_audio, sampling_rate=16000, return_tensors='pt', padding=True)
with torch.no_grad():
    logits = model(inputs.input_values).logits
pred_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(pred_ids)[0]
print("Transcription with noise:", transcription)

# Now harder synthetic noise
audio, _ = librosa.load('robert.wav', sr=16000)
# Create Gaussian white noise with mean 0, standard deviation 0.02, same length as the audio
noise = np.random.normal(0, 0.05, audio.shape)
# Add the noise directly to the original audio to simulate a noisy environment
noisy_audio = audio + noise

inputs = processor(noisy_audio, sampling_rate=16000, return_tensors='pt', padding=True)
with torch.no_grad():
    logits = model(inputs.input_values).logits
pred_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(pred_ids)[0]
print("Transcription with harder noise:", transcription)


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Transcription with noise: OE MOR IMROVER AND I THINK YOU THE MOTHER SHOULD HAVE GREAT CURIOS BEPORI E VO DON ER ON
Transcription with harder noise: ER ANOVER AN ITINATER AETORTORTORON


In [17]:
# Configs 
import matplotlib.pyplot as plt
model = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-large-960h-lv60-self')


# Print a summary of the model's architecture
print(model)
# Trainable params
num_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {num_params:,}")
# Memory footprint
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
# Specific layer output
audio, _ = librosa.load('child.wav', sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors='pt', padding=True)

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)
    # hidden_states: a tuple, each item is the output at a layer
    last_hidden = outputs.hidden_states[-1]

print("Last hidden state shape:", last_hidden.shape)
# Output pretransformer
features = model.wav2vec2.feature_extractor(inputs.input_values)
print("Feature extractor output shape:", features.shape)


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec


## 🧑‍🏫 8. Results Table & Error Analysis

| Metric | MyST | PFSTAR | CMU-Kids |
|--------|------|--------|----------|
| wav2vec2 | 7.42 | 2.91 | 12.77 |
| Whisper | 12.28 | 2.98 | 15.08 |

### Word Error Rate (WER) Formula

$$WER = \frac{S + D + I}{N} \times 100$$

**Components:**
- **S** = Substitutions (words replaced incorrectly)
- **D** = Deletions (words missing from transcription)
- **I** = Insertions (extra words added incorrectly)
- **N** = Total number of words in reference (ground truth)

**Example:**

Reference: "the cat sat on the mat"  
Hypothesis: "the dog sat the mat"

- S = 1 (cat → dog)
- D = 1 (missing "on")
- I = 0 (no extra words)
- N = 6 (total words)

$$WER = \frac{1 + 1 + 0}{6} \times 100 = 33.3\%$$

**Interpretation:**
- Lower is better (0% = perfect)
- Adult ASR: ~2-3% WER
- Child ASR: ~12-15% WER ← Research gap

---


| Category             | Wav2Vec2                                                                       | Whisper                                                                          |
| -------------------- | ------------------------------------------------------------------------------ | -------------------------------------------------------------------------------- |
| Developer            | Meta (Facebook AI Research)                                                    | OpenAI                                                                           |
| Main Function        | Speech-to-text (ASR)                                                           | Speech-to-text, translation, language ID                                         |
| Architecture         | Encoder-only transformer; CTC or seq2seq decoder                               | Encoder-decoder transformer; generative, autoregressive decoding                 |
| Input Feature        | Raw audio or extracted features (e.g. MFCC)                                    | Log-Mel spectrogram                                                              |
| Training Data        | English (base), multilingual (XLSR)                                            | 680,000+ hours, 99 languages, noisy & accented                                   |
| Multilingual         | XLS-R models support many languages, but less robust than Whisper              | Highly robust, supports many languages out-of-the-box                            |
| Speed                | Fast (suitable for real-time)                                                  | Slower (especially larger models)                                                |
| Accuracy             | Best for English, medical, legal, call center; achieves WER <5% on LibriSpeech | Often achieves lower WER on diverse data, best on noisy conditions, multilingual |
| Robustness           | Sensitive to data quality, less robust to noise                                | Very robust to background noise, accents, downsampling                           |
| Resource Usage       | Efficient, runs on consumer hardware                                           | Large models need high-end GPUs, high VRAM                                       |
| Special Features     | Can be fine-tuned for specific domains                                         | Handles translation, timestamping, language ID                                   |
| Real-world Benchmark | Excels in English-only scenarios                                               | Best for multilingual, diverse, noisy scenarios                                  |

## 📈 9. Research Methodology & Timeline
![research_workflow.png](charts/research_workflow.png)
---

## 🚀 10. Roadmap, Next Steps & Future Work

- Short-term: dataset curation, initial training
- Medium: architecure/models adaptation
- Long-term: deployment, educational pilots, publications
- Long-long term: Romanian corpus, transfer learning from English corpus

---
## 📝 Notes & ToDo
- [ ] Atleast start :)
- [ ] Deep Learning from scratch
- [ ] Understand speech techniques under transfer learning from images
- [ ] State-of-the-art ideas
- [ ] State-of-the-art implementation
- [ ] Start with English
- [ ] Fine tuning
- [ ] Transfer to Romanian if possible (NTH)
- [ ] Deploy integrated model for testing 


---
## 📞 Contact & Utils
https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.81908&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false

https://huggingface.co/docs/transformers/model_doc/wav2vec2

wav2vec2 - https://arxiv.org/pdf/2006.11477

whisper - https://huggingface.co/spaces/openai/whisper

3blue1brown - https://www.3blue1brown.com/?v=neural-networks

Robert-Florin Simion | simion.robert.florin@gmail.com | UPB ETTI 
| Advisor: Prof. Dr. Ing. Dragoș Burileanu | robert-florin.simion@ing.com
